# 🚀 Process Monitor Dashboard

Real-time system process management interface.

In [2]:
# === CONSOLIDATED PROCESS MONITOR DASHBOARD ===
import os
import sys
import time
import threading
import subprocess

# Global flag to prevent duplicate background threads across cell re-runs
if '_pm_running' not in globals():
    global _pm_running
    _pm_running = {'val': False}
else:
    # Stop any existing background loop before creating a new dashboard
    _pm_running['val'] = False
    time.sleep(0.5)  # Give the old thread a moment to exit

def setup_and_run():
    print("📦 Checking environment...")
    
    # 1. Auto-install missing dependencies
    try:
        import pandas as pd
        import ipywidgets as widgets
        from IPython.display import display, clear_output, HTML
    except ImportError:
        print("⚠️ Missing libraries. Installing pandas and ipywidgets...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "ipywidgets"])
        print("✅ Installed. PLEASE RESTART KERNEL AND RUN AGAIN.")
        return

    # Clear previous output to prevent widget stacking
    try: clear_output(wait=True)
    except: pass

    # 2. Path Setup
    project_root = os.path.abspath(".")
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

    try:
        from gui.notebook_interface import fetch_processes, processes_to_dataframe, filter_processes, group_processes, sort_processes, kill_process, pause_process, resume_process, set_priority, _style_dataframe
        print("🔌 Backend bridge connected.")
    except Exception as e:
        print(f"❌ Fatal Error: Could not load backend bridge: {e}")
        return

    # --- Logic Helpers ---
    PROTECTED_PROCESSES = {
        "system", "svchost.exe", "csrss.exe", "wininit.exe", 
        "services.exe", "smss.exe", "lsass.exe", "explorer.exe"
    }

    def is_safe_to_modify(pid, name):
        if pid <= 4: return False
        if name.lower() in PROTECTED_PROCESSES: return False
        return True

    # --- Auto Control State ---
    auto_history = {}

    # --- UI Components ---
    refresh_interval = 5

    dropdown = widgets.Dropdown(description="Process:", layout={'width': '500px'})
    kill_btn = widgets.Button(description="Kill", button_style='danger', icon='trash')
    pause_btn = widgets.Button(description="Pause", icon='pause')
    resume_btn = widgets.Button(description="Resume", icon='play')
    priority_slider = widgets.IntSlider(min=-10, max=10, value=0, description="Priority")
    set_priority_btn = widgets.Button(description="Set Priority", button_style='info')
    auto_toggle = widgets.ToggleButton(value=False, description="Auto Control: OFF", button_style='warning', icon='robot')
    status_label = widgets.Label(value="Status: Ready")
    start_btn = widgets.Button(description="Start Dashboard", button_style='success', icon='play-circle')
    stop_btn = widgets.Button(description="Stop Dashboard", button_style='danger', icon='stop-circle')
    
    # Use HTML widget instead of Output widget for thread safety
    table_html = widgets.HTML(value="<p>Click 'Start Dashboard' to load processes...</p>")

    def on_auto_toggle(change):
        if change['new']:
            auto_toggle.description = "Auto Control: ON"
            auto_toggle.button_style = "success"
        else:
            auto_toggle.description = "Auto Control: OFF"
            auto_toggle.button_style = "warning"
            
    auto_toggle.observe(on_auto_toggle, names='value')


    # --- Layout ---
    header = widgets.HTML("""
        <div style='background-color: #2e86de; padding: 15px; border-radius: 10px; margin-bottom: 20px;'>
            <h1 style='color: white; margin: 0;'>🚀 Process Monitor</h1>
            <p style='color: #dff9fb; margin: 5px 0 0 0;'>Real-time system process management</p>
        </div>
    """)
    
    controls = widgets.VBox([
        widgets.HBox([start_btn, stop_btn]),
        dropdown,
        widgets.HBox([kill_btn, pause_btn, resume_btn]),
        widgets.HBox([priority_slider, set_priority_btn]),
        auto_toggle,
        status_label
    ], layout={'padding': '15px', 'border': '1px solid #ddd', 'border-radius': '8px', 'margin-bottom': '20px'})

    dashboard_ui = widgets.VBox([header, controls, table_html])

    # --- Logic ---
    def update_display():
        try:
            processes = fetch_processes()
            df = processes_to_dataframe(processes)
            df = filter_processes(df)
            df = group_processes(df)
            df = sort_processes(df)
            
            if not df.empty:
                current_val = dropdown.value
                
                # Build new options
                new_options = [
                    (f"{row['Name']} (Instances: {len(row['PIDs']) if 'PIDs' in row else 1}) - CPU {row['CPU']}%", (tuple(row['PIDs']) if 'PIDs' in row else (row['PID'],), row['Name']))
                    for _, row in df.iterrows()
                ]
                dropdown.options = new_options
                
                # Restore previous selection if it still exists
                if current_val:
                    for opt in new_options:
                        if opt[1] == current_val:
                            dropdown.value = current_val
                            break
            
            # Update HTML value directly (thread-safe in ipywidgets)
            try:
                table_html.value = _style_dataframe(df).to_html()
            except Exception as e:
                table_html.value = df.to_html()
            
            if auto_toggle.value:
                action_taken = False
                for _, row in df.iterrows():
                    name = row["Name"]
                    pids = row["PIDs"] if "PIDs" in row else [row["PID"]]
                    
                    if row["CPU"] > 40.0:
                        auto_history[name] = auto_history.get(name, 0) + 1
                        ticks = auto_history[name]
                        if ticks == 1:
                            for p in pids:
                                if is_safe_to_modify(p, name): set_priority(p, 5)
                            status_label.value = f"Auto: Lowered priority of {name} (CPU: {row['CPU']}%)"
                            action_taken = True
                        elif ticks > 3:
                            for p in pids:
                                if is_safe_to_modify(p, name): pause_process(p)
                            status_label.value = f"Auto: Paused {name} (CPU persisted > 40%)"
                            action_taken = True
                    else:
                        if name in auto_history:
                            auto_history[name] = max(0, auto_history[name] - 1)
                
                if not action_taken and status_label.value.startswith("Status: Running"):
                    status_label.value = "Status: Running... (Auto Control Monitoring)"
            else:
                if status_label.value.startswith("Status: Running... (Auto Control"):
                    status_label.value = "Status: Running..." 
        except Exception as e:
            status_label.value = f"Status: Update Error - {e}"

    def on_action(b):
        if not dropdown.value: return
        pids, name = dropdown.value
        
        for pid in pids:
            if not is_safe_to_modify(pid, name):
                status_label.value = f"Status: Action Blocked! Safe mode prevented modifying {name} (PID: {pid})"
                continue
            try:
                if b.description == "Kill": kill_process(pid)
                elif b.description == "Pause": pause_process(pid)
                elif b.description == "Resume": resume_process(pid)
                elif b.description == "Set Priority": set_priority(pid, priority_slider.value)
            except Exception as e:
                status_label.value = f"Status: Action Error - {e}"
        status_label.value = f"Status: Executed {b.description} on {name} ({len(pids)} instances)" 
    def background_loop():
        while _pm_running['val']:
            update_display()
            time.sleep(refresh_interval)

    def start_loop(b):
        if _pm_running['val']: return
        _pm_running['val'] = True
        start_btn.disabled = True
        status_label.value = "Status: Running..."
        threading.Thread(target=background_loop, daemon=True).start()

    def stop_loop(b):
        _pm_running['val'] = False
        start_btn.disabled = False
        status_label.value = "Status: Stopped"

    # Bindings
    for btn in [kill_btn, pause_btn, resume_btn, set_priority_btn]:
        btn.on_click(on_action)
    start_btn.on_click(start_loop)
    stop_btn.on_click(stop_loop)

    # Initial display
    display(dashboard_ui)
    print("✅ Dashboard Loaded. Click 'Start Dashboard' to begin.")

setup_and_run()


🔌 Backend bridge connected.


✅ Dashboard Loaded. Click 'Start Dashboard' to begin.
